# spVIPES MIG Ablation Walkthrough

This notebook demonstrates the **Mutual Information Gap (MIG)** objective
introduced in spVIPES (inspired by CellDISECT and Multi-ContrastiveVAE) and
shows how to run systematic ablations over its components.

## What is MIG?

MIG enforces explicit information gaps between latent spaces: each latent
should be informative about its 'owned' factor and uninformative about all
others. For spVIPES with two factors (group identity and cell-type label):

```
MIG_shared  = I(z_shared;  label) - I(z_shared;  group)   -> maximize
MIG_private = I(z_private; group) - I(z_private; label)   -> maximize
```

These are realised through 4 auxiliary classifiers:

| Component | Latent | Target | Encoder direction | Needs labels? |
|---|---|---|---|---|
| `q_label_shared` | z_shared | label | preserve (no GRL) | yes |
| `q_group_shared` | z_shared | group | erase (GRL) | no |
| `q_group_private` | z_private | group | preserve (no GRL) | no |
| `q_label_private` | z_private | label | erase (GRL) | yes |

Plus an optional **prototype contrastive InfoNCE** on z_shared.

## What this notebook does

1. Loads the DIALOGUE example dataset (3 groups, labelled cells).
2. Trains spVIPES with each `mig_preset` and reports metrics.
3. Runs single-component ablations starting from `mig_preset='full'`.
4. Compares ablations on group-mixing (shared) and label-preservation (shared).

Each MIG component can be independently disabled by setting its weight to 0,
or activated via a named preset.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt

import pertpy as pt
import spVIPES
from spVIPES.model._mig_presets import MIG_PRESETS

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
sc.settings.set_figure_params(dpi=100, frameon=False)

# Inspect the available presets
for name, weights in MIG_PRESETS.items():
    print(f"{name}: {weights}")

## 1. Load and prepare the dataset

We use the DIALOGUE example: PBMCs annotated by cell type (`cell.subtypes`)
and clinical status (`clinical.status`). Groups are defined by
`clinical.status`; labels for label-based PoE are `cell.subtypes`.

In [ ]:
adata_full = pt.dt.dialogue_example()
sc.pp.normalize_total(adata_full)
sc.pp.log1p(adata_full)

# Split by clinical status into N groups
groups_dict = {
    str(s): adata_full[adata_full.obs["clinical.status"] == s].copy()
    for s in adata_full.obs["clinical.status"].unique()
}
for s, a in groups_dict.items():
    print(f"  {s}: {a.n_obs} cells")

adata = spVIPES.data.prepare_adatas(groups_dict)
spVIPES.model.spVIPES.setup_anndata(
    adata, groups_key="groups", label_key="cell.subtypes"
)

## 2. Helper: train a model with a given MIG configuration

In [ ]:
def train_and_score(adata, *, max_epochs=50, **mig_kwargs):
    """Train a spVIPES model with given MIG kwargs and return latent + history."""
    model = spVIPES.model.spVIPES(
        adata,
        n_dimensions_shared=20,
        n_dimensions_private=8,
        **mig_kwargs,
    )
    model.train(max_epochs=max_epochs, batch_size=256)
    z = model.get_latent_representation()
    return model, z

def kbet_like(z_shared, group_labels, k=20):
    """Cheap proxy for kBET: average label distribution among k-NN neighbours.
    Higher = better mixing across groups."""
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k+1).fit(z_shared)
    _, idx = nn.kneighbors(z_shared)
    idx = idx[:, 1:]  # drop self
    g = np.asarray(group_labels)
    expected = pd.Series(g).value_counts(normalize=True).reindex(np.unique(g)).values
    chi = []
    for i in range(len(g)):
        observed = pd.Series(g[idx[i]]).value_counts(normalize=True).reindex(np.unique(g), fill_value=0).values
        chi.append(((observed - expected) ** 2 / (expected + 1e-9)).sum())
    return float(np.exp(-np.mean(chi)))  # in (0, 1]; closer to 1 = better mixing

def label_purity(z_shared, labels, k=20):
    """Fraction of k-NN neighbours sharing the cell's label.
    Higher = better cell-type clustering in z_shared."""
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k+1).fit(z_shared)
    _, idx = nn.kneighbors(z_shared)
    idx = idx[:, 1:]
    l = np.asarray(labels)
    return float(np.mean([(l[idx[i]] == l[i]).mean() for i in range(len(l))]))

## 3. Compare presets

We train one model per preset and tabulate group mixing (kBET-like) vs.
label preservation (k-NN purity) on the **shared** latent.

In [ ]:
results = {}
for preset in ["off", "shared_only", "private_only", "adversarial_only",
               "supervised_only", "no_contrastive", "full"]:
    print(f"\n=== preset={preset} ===")
    model, z = train_and_score(adata, mig_preset=preset)
    z_shared = z["shared"]
    groups = adata.obs["groups"].values
    labels = adata.obs["cell.subtypes"].values
    results[preset] = {
        "group_mixing_shared": kbet_like(z_shared, groups),
        "label_purity_shared": label_purity(z_shared, labels),
    }

preset_df = pd.DataFrame(results).T
preset_df

## 4. Single-component ablations (starting from `full`)

For each MIG component, set its weight to 0 and re-train. The drop in
metrics tells you what that component contributes.

In [ ]:
ablate_components = [
    "mig_group_shared_weight",
    "mig_label_shared_weight",
    "mig_group_private_weight",
    "mig_label_private_weight",
    "contrastive_weight",
]

ablation_results = {}
for component in ablate_components:
    label = f"full minus {component}"
    print(f"\n=== {label} ===")
    model, z = train_and_score(
        adata, mig_preset="full", **{component: 0.0}
    )
    z_shared = z["shared"]
    groups = adata.obs["groups"].values
    labels = adata.obs["cell.subtypes"].values
    ablation_results[label] = {
        "group_mixing_shared": kbet_like(z_shared, groups),
        "label_purity_shared": label_purity(z_shared, labels),
    }

ablation_df = pd.DataFrame(ablation_results).T
ablation_df

## 5. Visual summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

preset_df.plot.bar(ax=axes[0])
axes[0].set_title("MIG presets: shared-latent metrics")
axes[0].set_ylabel("score (higher is better)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend()

ablation_df.plot.bar(ax=axes[1])
axes[1].set_title("Single-component ablations from full")
axes[1].set_ylabel("score (higher is better)")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Interpreting the results

**Expected patterns:**

- `mig_preset='full'` should produce the highest combined score: high group
  mixing in z_shared (`q_group_shared` erases group identity) plus high label
  preservation (`q_label_shared` + contrastive enforce cell-type clustering).
- Removing `mig_group_shared_weight` should drop **group mixing** the most.
- Removing `mig_label_shared_weight` or `contrastive_weight` should drop
  **label purity** the most.
- `private_only` should leave the shared latent unchanged from `off` since
  it only constrains the private latents.

## 7. Custom configurations

Combine a preset with overrides for any custom configuration:

```python
# Strong adversarial group erasure with weak label supervision
model = spVIPES.model.spVIPES(
    adata,
    mig_preset="full",
    mig_group_shared_weight=2.0,    # boost adversarial
    mig_label_shared_weight=0.5,    # weaken supervision
    contrastive_weight=0.0,         # disable contrastive
)
```

**Validation:** if labels are not registered, setting `mig_label_*_weight > 0`
or `contrastive_weight > 0` raises a clear `ValueError` at construction time.
Group classifiers (`q_group_shared`, `q_group_private`) work without labels.